##Day 6 — DMRG: Variational Optimization of an MPS

###Core idea

DMRG finds a low-energy / ground-state approximation by optimizing an MPS locally instead of diagonalizing the full Hamiltonian.

###Key relations

Ground-state equation:
H|ψ₀⟩ = E₀|ψ₀⟩

Variational energy:
E[ψ] = ⟨ψ|H|ψ⟩ / ⟨ψ|ψ⟩

For a normalized state:
E[ψ] = ⟨ψ|H|ψ⟩

Variational principle:
E[ψ] ≥ E₀

###Local optimization

Keep most MPS tensors fixed and optimize one or two neighboring tensors using an effective Hamiltonian.

H_eff θ = E_local θ

In this notebook, we perform two local optimization steps on a three-site chain.

###Environment

A fixed part of the MPS contributes through effective operators such as

L_O = A† O A

###SVD update

θ → U Σ V†

Singular values = Schmidt coefficients (for a normalized state in Schmidt form)

Schmidt rank:
χ = number of nonzero singular values

Exact bond dimension:
D = χ

No intentional truncation is performed in this notebook.

###Local optimization steps

We optimize sites 1–2, then sites 2–3. A complete left-to-right and right-to-left DMRG sweep and convergence loop are not implemented.

###Validation

Residual:
r = H|ψ⟩ − E|ψ⟩

A small residual indicates that the final state is close to an eigenstate. We also compare its energy with the exact ground-state energy.

###Mental map

Hamiltonian → initial MPS → local optimization → effective Hamiltonian → SVD → updated MPS → final reconstruction → energy and residual checks → comparison with exact ground-state energy

In [ ]:
import numpy as np

n_sites = 3

psi_initial = np.zeros(2**n_sites, dtype=complex)
psi_initial[0] = 1.0

print("Shape:", psi_initial.shape)
print("Norm:", np.linalg.norm(psi_initial))

Shape: (8,)
Norm: 1.0


In [ ]:
A1 = np.array([[[1], [0]]], dtype=complex)
A2 = np.array([[[1], [0]]], dtype=complex)
A3 = np.array([[[1], [0]]], dtype=complex)

print("A1 shape:", A1.shape)
print("A2 shape:", A2.shape)
print("A3 shape:", A3.shape)

A1 shape: (1, 2, 1)
A2 shape: (1, 2, 1)
A3 shape: (1, 2, 1)


In [ ]:
I = np.eye(2, dtype=complex)

X = np.array([[0, 1],
[1, 0]], dtype=complex)

Y = np.array([[0, -1j],
[1j, 0]], dtype=complex)

Z = np.array([[1, 0],
[0, -1]], dtype=complex)

H12 = (np.kron(np.kron(X, X), I)
+ np.kron(np.kron(Y, Y), I)
+ np.kron(np.kron(Z, Z), I))

H23 =(np.kron(I, np.kron(X, X))
+ np.kron(I, np.kron(Y, Y))
+ np.kron(I, np.kron(Z, Z)))

H = H12 + H23

print("Hamiltonian shape:", H.shape)

Hamiltonian shape: (8, 8)


In [ ]:
psi = psi_initial

energy = np.vdot(psi, H @ psi)

print("Energy:", energy)

Energy: (2+0j)


In [ ]:
theta = np.tensordot(A1, A2, axes=([2], [0]))

print("Theta shape:", theta.shape)

Theta shape: (1, 2, 2, 1)


In [ ]:
theta_vector = theta.reshape(-1)

print("Theta vector shape:", theta_vector.shape)
print("Theta vector:", theta_vector)

Theta vector shape: (4,)
Theta vector: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]


In [ ]:
H_eff = (
np.kron(X, X)
+ np.kron(Y, Y)
+ np.kron(Z, Z)
+ np.kron(I, Z)
)
print("Effective Hamiltonian shape:", H_eff.shape)


Effective Hamiltonian shape: (4, 4)


In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(H_eff)

local_energy = eigenvalues[0]
theta_vector_new = eigenvectors[:, 0]

print("Local energy:", local_energy)
print("New theta shape:", theta_vector_new.shape)

Local energy: -3.23606797749979
New theta shape: (4,)


In [ ]:
theta_new = theta_vector_new.reshape(1, 2, 2, 1)

print("Theta new shape:", theta_new.shape)

Theta new shape: (1, 2, 2, 1)


In [ ]:
theta_matrix = theta_new.reshape(2, 2)

print("Theta matrix shape:", theta_matrix.shape)
print("Theta matrix:")
print(theta_matrix)

Theta matrix shape: (2, 2)
Theta matrix:
[[ 0.        +0.j -0.85065081+0.j]
 [ 0.52573111+0.j  0.        +0.j]]


In [ ]:
U, S, Vh = np.linalg.svd(theta_matrix, full_matrices=False)

print("U shape:", U.shape)
print("Singular values:", S)
print("Vh shape:", Vh.shape)

U shape: (2, 2)
Singular values: [0.85065081 0.52573111]
Vh shape: (2, 2)


In [ ]:
D_new = len(S)

A1_new = U.reshape(1, 2, D_new)

print("D_new:", D_new)
print("A1_new shape:", A1_new.shape)

D_new: 2
A1_new shape: (1, 2, 2)


In [ ]:
A2_matrix = np.diag(S) @ Vh

A2_new = A2_matrix.reshape(D_new, 2, 1)

print("A2_new shape:", A2_new.shape)

A2_new shape: (2, 2, 1)


In [ ]:
M12_new = np.tensordot(
A1_new,
A2_new,
axes=([2], [0])
)
M123_new = np.tensordot(
M12_new,
A3,
axes=([3], [0])
)

psi_new = np.squeeze(M123_new).reshape(-1)

print("New state shape:", psi_new.shape)
print("New state norm:", np.linalg.norm(psi_new))


New state shape: (8,)
New state norm: 0.9999999999999999


In [ ]:
energy_new = np.vdot(psi_new, H @ psi_new)

print("New total energy:", energy_new)

New total energy: (-3.236067977499789+0j)


In [ ]:
A1_matrix = A1_new[0, :, :]

print("A1 matrix shape:", A1_matrix.shape)

A1 matrix shape: (2, 2)


In [ ]:
L_X = A1_matrix.conj().T @ X @ A1_matrix

print("L_X shape:", L_X.shape)
print("L_X:")
print(L_X)

L_X shape: (2, 2)
L_X:
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]


In [ ]:
L_Y = A1_matrix.conj().T @ Y @ A1_matrix

print("L_Y shape:", L_Y.shape)
print("L_Y:")
print(L_Y)

L_Y shape: (2, 2)
L_Y:
[[0.+0.j 0.-1.j]
 [0.+1.j 0.+0.j]]


In [ ]:
L_Z = A1_matrix.conj().T @ Z @ A1_matrix

print("L_Z shape:", L_Z.shape)
print("L_Z:")
print(L_Z)

L_Z shape: (2, 2)
L_Z:
[[ 1.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j]]


In [ ]:
D_left = A1_new.shape[2]
I_D = np.eye(D_left, dtype=complex)

print("I_D shape:", I_D.shape)

I_D shape: (2, 2)


In [ ]:
H_eff_12 = (
np.kron(np.kron(L_X, X), I)
+ np.kron(np.kron(L_Y, Y), I)
+ np.kron(np.kron(L_Z, Z), I)
)
print("H_eff_12 shape:", H_eff_12.shape)

H_eff_12 shape: (8, 8)


In [ ]:
H_eff_23 = (
np.kron(np.kron(I_D, X), X)
+ np.kron(np.kron(I_D, Y), Y)
+ np.kron(np.kron(I_D, Z), Z)
)
print("H_eff_23 shape:", H_eff_23.shape)


H_eff_23 shape: (8, 8)


In [ ]:
H_eff = H_eff_12 + H_eff_23

print("Effective Hamiltonian shape:", H_eff.shape)

Effective Hamiltonian shape: (8, 8)


In [ ]:
eigenvalues_23, eigenvectors_23 = np.linalg.eigh(H_eff)

local_energy_23 = eigenvalues_23[0]
theta_vector_23 = eigenvectors_23[:, 0]

print("Local energy:", local_energy_23)
print("Theta vector shape:", theta_vector_23.shape)

Local energy: -4.0
Theta vector shape: (8,)


In [ ]:
theta_23 = theta_vector_23.reshape(2, 2, 2, 1)

print("Theta 23 shape:", theta_23.shape)

Theta 23 shape: (2, 2, 2, 1)


In [ ]:
theta_matrix_23 = theta_23.reshape(4, 2)

print("Theta matrix 23 shape:", theta_matrix_23.shape)

Theta matrix 23 shape: (4, 2)


In [ ]:
U_23, S_23, Vh_23 = np.linalg.svd(
theta_matrix_23,
full_matrices=False)
print("U shape:", U_23.shape)
print("Singular values:", S_23)
print("Vh shape:", Vh_23.shape)

U shape: (4, 2)
Singular values: [0.91287093 0.40824829]
Vh shape: (2, 2)


In [ ]:
D_new_23 = len(S_23)

A2_new_23 = U_23.reshape(2, 2, D_new_23)

print("D_new_23:", D_new_23)
print("A2_new_23 shape:", A2_new_23.shape)

D_new_23: 2
A2_new_23 shape: (2, 2, 2)


In [ ]:
A3_matrix_23 = np.diag(S_23) @ Vh_23

print("A3 matrix shape:", A3_matrix_23.shape)

A3 matrix shape: (2, 2)


In [ ]:
A3_new_23 = A3_matrix_23.reshape(D_new_23, 2, 1)

print("A3_new_23 shape:", A3_new_23.shape)

A3_new_23 shape: (2, 2, 1)


In [ ]:
M23_new = np.tensordot(
  A2_new_23,
  A3_new_23,
  axes=([2], [0])
  )
M123_new = np.tensordot(
  A1_new,
  M23_new,
  axes=([2], [0])
  )
psi_final = np.squeeze(M123_new).reshape(-1)

print("Final state shape:", psi_final.shape)
print("Final state norm:", np.linalg.norm(psi_final))

Final state shape: (8,)
Final state norm: 1.0


In [ ]:
energy_final = np.vdot(psi_final, H @ psi_final)

print("Final total energy:", energy_final)

Final total energy: (-4+0j)


In [ ]:
residual = H @ psi_final - energy_final * psi_final

print("Residual norm:", np.linalg.norm(residual))

Residual norm: 1.2329641936283707e-15


In [ ]:
exact_energies = np.linalg.eigvalsh(H)
exact_ground_energy = exact_energies[0]

print("Exact ground-state energy:", exact_ground_energy)
print("Final energy from current method:", energy_final)
print("Energy difference:", abs(energy_final - exact_ground_energy))

Exact ground-state energy: -4.0
Final energy from current method: (-4+0j)
Energy difference: 0.0
